In [1]:
import os
import sys
import joblib
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import wilcoxon, friedmanchisquare
 
sys.path.insert(0, '../../..')  # Load Parent directory to import utils
from utils.constants import Configs

RESULTS_DIR = Configs._wrist_only_loso_results_cache_dir

In [2]:
def load_all_fold_results(results_dir):
    rows = []
    for file in glob.glob(os.path.join(results_dir, "*.pkl")):
        obj = joblib.load(file)
        if isinstance(obj, dict) and "Fold_History" in obj:
            df = pd.DataFrame(obj["Fold_History"])

            # Flatten the metrics column if it exist
            if "metrics" in df.columns:
                metrics_df = pd.json_normalize(df.pop("metrics"))
                df = pd.concat([df, metrics_df], axis=1)
                
            stem = Path(file).stem.replace("wrist_only_loso_results_", "")
            model, representation, phase = stem.rsplit("_", 2)
            df["Model"] = model
            df["Representation"] = representation
            df["Phase"] = phase
            rows.append(df)
        elif isinstance(obj, pd.DataFrame):
            rows.append(obj)
        else:
            raise TypeError(f"{file} contains {type(obj).__name__}, not a supported result format")
    return pd.concat(rows, ignore_index=True)


In [3]:
consolidated_data = load_all_fold_results(RESULTS_DIR)
print(f"Successfully loaded {len(consolidated_data)} fold records.")
consolidated_data.head()

Successfully loaded 418 fold records.


,subject_out,params,AUPRC,AUROC,TP,FP,FN,TN,Precision,Recall,F1,F2,MCC,Pct_Rejection,N_Known_Positives,N_Predicted_Positives,Scores,Model,Representation,Phase
0,1,{'z_threshold': 2.0},NaN,NaN,0,0,0,5,NaN,NaN,NaN,NaN,NaN,0.0,0,0,None,ZScore,ShapeBased,Move
1,2,{'z_threshold': 2.0},NaN,NaN,1,0,0,4,1.0,1.0,1.0,1.0,1.0,20.0,1,1,None,ZScore,ShapeBased,Move
2,3,{'z_threshold': 2.0},NaN,NaN,0,0,0,5,NaN,NaN,NaN,NaN,NaN,0.0,0,0,None,ZScore,ShapeBased,Move
3,4,{'z_threshold': 2.0},NaN,NaN,1,0,0,4,1.0,1.0,1.0,1.0,1.0,20.0,1,1,None,ZScore,ShapeBased,Move
4,5,{'z_threshold': 2.5},NaN,NaN,0,0,0,5,NaN,NaN,NaN,NaN,NaN,0.0,0,0,None,ZScore,ShapeBased,Move


In [4]:
def _get_valid_models_and_subjects(df_filtered, models_list, baseline_model, metric_col):
    """Return models with valid F2 values and their valid subjects."""
    per_model_valid_subjects = {}

    for model in models_list:
        model_data = df_filtered[
            df_filtered['Model'] == model
        ].dropna(subset=[metric_col])

        if not model_data.empty:
            per_model_valid_subjects[model] = set(
                model_data['subject_out']
            )

    if baseline_model not in per_model_valid_subjects:
        return None, None

    valid_models = list(per_model_valid_subjects.keys())

    if len(valid_models) < 2:
        return None, None

    return valid_models, per_model_valid_subjects

def _calculate_friedman_test(valid_models, aligned_scores):
    if len(valid_models) > 2:
        stat_f, p_friedman = friedmanchisquare(*aligned_scores)
        friedman_str = f"p={p_friedman:.4f}" if p_friedman >= 0.0001 else "p<0.0001"
        friedman_sig = p_friedman < 0.05
    else:
        friedman_str, friedman_sig = "N/A", False
    return friedman_str,friedman_sig

def _calculate_wilcoxon_tests(friedman_sig, baseline_scores, baseline_median, k, alpha_corrected, model_scores, model_median):
    if friedman_sig:
        stat_w, p_wilcoxon = wilcoxon(model_scores, baseline_scores, alternative='two-sided')
        wilcox_str = f"{p_wilcoxon:.4f}" if p_wilcoxon >= 0.0001 else "<0.0001"

        # Check if the model is superior to the baseline using both uncorrected and corrected p-values
        is_superior_uncorrected = (p_wilcoxon < 0.05) and (model_median > baseline_median)
        is_superior_corrected = (p_wilcoxon < alpha_corrected) and (model_median > baseline_median)
 
        uncorrected_str = "Yes" if is_superior_uncorrected else "No"
        superior_str = "Yes" if is_superior_corrected else "No"
        alpha_str = f"{alpha_corrected:.4f} (k={k})"
    else:
        wilcox_str = "Skipped (Friedman > 0.05)"
        uncorrected_str = "-"
        superior_str = "-"
        alpha_str = "-"
    return wilcox_str,uncorrected_str,superior_str,alpha_str

def _calculate_bonferroni_dunn(baseline_model, valid_models):
    """
    Bonferroni-Dunn correction for multiple comparisons
    """
    non_baseline_models = [m for m in valid_models if m != baseline_model]
    k = len(non_baseline_models)
    alpha_corrected = 0.05 / k if k > 0 else 0.05
    return k,alpha_corrected

In [5]:
def export_statistical_significance_csv(df, phases, representations, baseline_model, models_list, output_csv, metric_col='test_f2', metric_label='F2',
                                          min_subjects=5):
    """
    Execute friedman and wilcoxon tests with Bonferroni-Dunn correction. And export a table ready for the papper.
    """

    is_mcc = (metric_col == 'MCC')
    scale = 1 if is_mcc else 100
    unit_label = metric_label if is_mcc else f'{metric_label} (%)'
 
    results = []
    for rep in representations:
        for phase in phases:

            # Place was not computed for any model, so I cannot make any comparison
            if phase == 'Place':
                results.append({
                    'Representation': rep, 'Phase': phase, 'Model': '-',
                    f'Median {unit_label}': '-', 'Friedman (Global)': 'N/A (no positives in Place)',
                    'Wilcoxon (vs Baseline)': '-', 'Alpha (Bonferroni-Dunn)': '-',
                    'Significant (uncorrected)': '-', 'Significant Superiority': '-',
                    'N Subjects Used': 0,
                })
                continue
 
            df_filtered = df[(df['Phase'] == phase) & (df['Representation'] == rep)]
            if df_filtered.empty:
                continue
 
            baseline_data = df_filtered[df_filtered['Model'] == baseline_model].sort_values('subject_out')
            if baseline_data.empty:
                continue
 
            # --- Determinar o conjunto de modelos válidos e recolher os subject_out não-NaN de cada um ---
            valid_models, per_model_valid_subjects = _get_valid_models_and_subjects(
                df_filtered, models_list, baseline_model, metric_col
            )

            if valid_models is None:
                print(f"⚠️ {rep}/{phase}: baseline '{baseline_model}' sem F2 válido, ou menos de 2 modelos "
                    f"comparáveis nesta combinação -- a saltar.")
                results.append({
                    'Representation': rep, 'Phase': phase, 'Model': '-',
                    f'Median {unit_label}': '-', 'Friedman (Global)': 'N/A (baseline missing or <2 models)',
                    'Wilcoxon (vs Baseline)': '-', 'Alpha (Bonferroni-Dunn)': '-',
                    'Significant (uncorrected)': '-', 'Significant Superiority': '-',
                    'N Subjects Used': 0,
                })
                continue
            
            # --- Interseção: sujeitos com F2 não-NaN em TODOS os modelos válidos desta combinação ---
            common_subjects = set.intersection(*per_model_valid_subjects.values())
            n_common = len(common_subjects)
 
            if n_common < min_subjects:
                print(f"⚠️ Apenas {n_common} sujeitos com F2 válido em todos os modelos "
                      f"({rep}/{phase}) -- abaixo do mínimo ({min_subjects}). Marcado como N/A.")
                for model in valid_models:
                    results.append({
                        'Representation': rep, 'Phase': phase, 'Model': model,
                        f'Median {unit_label}': '-', 'Friedman (Global)': 'N/A (too few subjects)',
                        'Wilcoxon (vs Baseline)': '-', 'Alpha (Bonferroni-Dunn)': '-',
                        'Significant (uncorrected)': '-', 'Significant Superiority': '-',
                        'N Subjects Used': n_common,
                    })
                continue
 
            common_subjects_sorted = sorted(common_subjects)
 
            def get_aligned_scores(model, metric_column=metric_col):
                m_data = df_filtered[(df_filtered['Model'] == model) &
                                      (df_filtered['subject_out'].isin(common_subjects))]
                m_data = m_data.set_index('subject_out').loc[common_subjects_sorted]
                return m_data[metric_column].values
 
            aligned_scores = [get_aligned_scores(m) for m in valid_models]

            # Execute Friedman test if there are more than 2 valid models
            friedman_str, friedman_sig = _calculate_friedman_test(valid_models, aligned_scores)
            
            baseline_scores = get_aligned_scores(baseline_model)
            baseline_median = np.median(baseline_scores) * scale

            k, alpha_corrected = _calculate_bonferroni_dunn(baseline_model, valid_models)

            # For each model, compute the median F2, and if applicable, the Wilcoxon test against the baseline
            for model in valid_models:
                if model == baseline_model:
                    results.append({
                        'Representation': rep, 'Phase': phase, 'Model': model,
                        f'Median {unit_label}': f"{baseline_median:.2f}",
                        'Friedman (Global)': friedman_str,
                        'Wilcoxon (vs Baseline)': "-", 'Alpha (Bonferroni-Dunn)': "-",
                        'Significant (uncorrected)': "-", 'Significant Superiority': "-",
                        'N Subjects Used': n_common,
                    })
                    continue
 
                model_scores = get_aligned_scores(model)
                model_median = np.median(model_scores) * scale
 
                wilcox_str, uncorrected_str, superior_str, alpha_str = _calculate_wilcoxon_tests(
                    friedman_sig, baseline_scores, baseline_median, k, alpha_corrected, model_scores, model_median
                )
 
                results.append({
                    'Representation': rep, 'Phase': phase, 'Model': model,
                    f'Median {unit_label}': f"{model_median:.2f}",
                    'Friedman (Global)': friedman_str,
                    'Wilcoxon (vs Baseline)': wilcox_str,
                    'Alpha (Bonferroni-Dunn)': alpha_str,
                    'Significant (uncorrected)': uncorrected_str,
                    'Significant Superiority': superior_str,
                    'N Subjects Used': n_common,
                })
 
    out_df = pd.DataFrame(results)
    out_df.to_csv(output_csv, index=False)
    print(f"✅ Statistical results exported to: {output_csv}")
    return out_df


In [6]:
models_all = ["ZScore", "IForest", "DBSCAN", "LOF", "OCSVM", "MatrixProfile"]
stats_table = export_statistical_significance_csv(
    df=consolidated_data,
    phases=['Pick', 'Move', 'Place'],
    representations=['ShapeBased', 'FeatureBased'],
    baseline_model="ZScore",
    models_list=models_all,
    output_csv="statistical_significance_table.csv",
    min_subjects=5,
    metric_col='MCC', # name should match de column in the metrics collumn in the fold_history dataframe
    metric_label='MCC'
)

✅ Statistical results exported to: statistical_significance_table.csv


In [7]:
stats_table

,Representation,Phase,Model,Median MCC,Friedman (Global),Wilcoxon (vs Baseline),Alpha (Bonferroni-Dunn),Significant (uncorrected),Significant Superiority,N Subjects Used
0,ShapeBased,Pick,ZScore,0.00,p=0.0389,-,-,-,-,9
1,ShapeBased,Pick,IForest,0.00,p=0.0389,0.8750,0.0100 (k=5),No,No,9
2,ShapeBased,Pick,DBSCAN,0.24,p=0.0389,0.0469,0.0100 (k=5),Yes,No,9
3,ShapeBased,Pick,LOF,0.06,p=0.0389,0.2188,0.0100 (k=5),No,No,9
4,ShapeBased,Pick,OCSVM,0.23,p=0.0389,0.0781,0.0100 (k=5),No,No,9
5,ShapeBased,Pick,MatrixProfile,0.00,p=0.0389,0.6250,0.0100 (k=5),No,No,9
6,ShapeBased,Move,ZScore,1.00,p=0.0485,-,-,-,-,5
7,ShapeBased,Move,IForest,0.00,p=0.0485,1.0000,0.0100 (k=5),No,No,5
8,ShapeBased,Move,DBSCAN,1.00,p=0.0485,0.5000,0.0100 (k=5),No,No,5
9,ShapeBased,Move,LOF,1.00,p=0.0485,1.0000,0.0100 (k=5),No,No,5
